# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Baseline Rule**

I will rank content items for review using two observable signals: content staleness and CTR opportunity. Content staleness is measured by the number of days since the content was last updated. CTR opportunity is identified using search impressions, average search position, and observed click-through rate. A higher score indicates a higher priority for human review. The rule is intended for decision support and prioritization, not as proof that updating a page will improve its future performance.

**Reason Codes**
- **STALE_CONTENT**: The content has not been updated for a long period and may deserve a refresh review.
- **LOW_CTR_OPPORTUNITY**: The content receives search impressions and has a reasonable average position but has relatively low CTR.
- **STALE_AND_LOW_CTR**: The content shows both staleness and a potential CTR opportunity.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

```
dim_content(
  client_hash_id
content_hash_id
content_updated_date
is_published
is_deleted
)
        │
        │ content_hash_id + client_hash_id
        ↓
fact_content_daily_performance(
  report_date
client_hash_id
content_hash_id
gsc_data_available
gsc_impressions
gsc_clicks
gsc_avg_position
)
```

### Calculating Baseline Action Score and Rank

I will now implement the logic to calculate the `baseline_action_score` based on 'content staleness' and 'CTR opportunity', and then rank the content items.

First, I'll create sample data for `dim_content` and `fact_content_daily_performance` to demonstrate the process. In a real scenario, these would be loaded from your data sources.

In [ ]:
import pandas as pd
import numpy as np
from datasets import load_dataset

# --- 1. Load Data from Hugging Face ---

print("Loading 'FlyRank/internship-warehouse' dataset configurations...")
try:
    # Load dim_content configuration
    print("Loading 'dim_content'...")
    dataset_dim_content = load_dataset('FlyRank/internship-warehouse', 'dim_content', split='train')
    df_dim_content = dataset_dim_content.to_pandas()
    print("df_dim_content loaded successfully. Head:")
    display(df_dim_content.head())

    # Load fact_content_daily_performance configuration
    print("\nLoading 'fact_content_daily_performance'...")
    dataset_fact_performance = load_dataset('FlyRank/internship-warehouse', 'fact_content_daily_performance', split='train')
    df_fact_performance = dataset_fact_performance.to_pandas()
    print("df_fact_performance loaded successfully. Head:")
    display(df_fact_performance.head())

except Exception as e:
    print(f"Error loading dataset configurations: {e}")
    print("Proceeding with synthetic data generation as a fallback.")

    # Fallback to previous synthetic data generation if Hugging Face load fails
    df_hf_sample = pd.DataFrame({'text': ['sample text ' + str(i) for i in range(100)], 'label': [i % 2 for i in range(100)]})
    df_dim_content = pd.DataFrame({
        'client_hash_id': df_hf_sample['label'].apply(lambda x: f'client_{x}'),
        'content_hash_id': ['hf_content_' + str(i) for i in range(len(df_hf_sample))],
        'content_updated_date': pd.to_datetime('2023-01-01') + pd.to_timedelta(np.arange(len(df_hf_sample)), unit='D'),
        'is_published': True,
        'is_deleted': False
    })

    num_performance_entries = len(df_dim_content) * 2
    data_fact_performance = {
        'report_date': pd.to_datetime('2024-05-20') + pd.to_timedelta(np.random.randint(0, 2, num_performance_entries), unit='D'),
        'client_hash_id': np.random.choice(df_dim_content['client_hash_id'].unique(), num_performance_entries),
        'content_hash_id': np.random.choice(df_dim_content['content_hash_id'].unique(), num_performance_entries),
        'gsc_data_available': True,
        'gsc_impressions': np.random.randint(100, 10000, num_performance_entries),
        'gsc_clicks': np.random.randint(1, 200, num_performance_entries),
        'gsc_avg_position': np.random.uniform(1.0, 50.0, num_performance_entries)
    }
    df_fact_performance = pd.DataFrame(data_fact_performance)

print("\nFinal df_dim_content head:")
display(df_dim_content.head())
print("\nFinal df_fact_performance head:")
display(df_fact_performance.head())

Loading 'FlyRank/internship-warehouse' dataset configurations...
Loading 'dim_content'...


Using the latest cached version of the dataset since FlyRank/internship-warehouse couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'dim_content' at /root/.cache/huggingface/datasets/FlyRank___internship-warehouse/dim_content/0.0.0/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2 (last modified on Fri Jul 31 16:33:34 2026).


df_dim_content loaded successfully. Head:


,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,...,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682.0,2555.0,None,None,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword_4329d7aede8e208b,url_3a66d2f2e36823ca,31,6,95,2026-06-12,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15438.0,2430.0,None,None,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword_9b08047d3d2a0406,url_809eda7a7e20b3b2,22,5,82,2026-05-09,2026-07-01,keyword article,...,4,2026-05-06,gemini-generate-content,gemini-3-flash-preview,16576.0,2645.0,None,None,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,keyword_e7cec7ab1804c1c2,url_5fb42bafc4399861,14,3,92,2026-06-15,2026-06-15,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15457.0,2522.0,None,None,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,keyword_56b0062a1d8b7524,url_ece0abc3e5fb75f9,24,6,98,2026-05-21,2026-06-01,keyword article,...,4,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15776.0,2552.0,None,None,True,False



Loading 'fact_content_daily_performance'...


Using the latest cached version of the dataset since FlyRank/internship-warehouse couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'fact_content_daily_performance' at /root/.cache/huggingface/datasets/FlyRank___internship-warehouse/fact_content_daily_performance/0.0.0/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2 (last modified on Fri Jul 31 16:36:44 2026).


Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

In [1]:
# --- 2. Pre-process and Aggregate Performance Data ---
# Ensure report_date is datetime type
df_fact_performance['report_date'] = pd.to_datetime(df_fact_performance['report_date'])

print(f"Shape of df_dim_content before merge: {df_dim_content.shape}")
print(f"Shape of df_fact_performance before merge: {df_fact_performance.shape}")

# For each content item, get the latest report date and aggregate performance metrics
df_agg_performance = df_fact_performance.groupby(['client_hash_id', 'content_hash_id']).agg(
    latest_report_date=('report_date', 'max'),
    total_impressions=('gsc_impressions', 'sum'),
    total_clicks=('gsc_clicks', 'sum'),
    avg_position=('gsc_avg_position', 'mean')
).reset_index()

# --- 3. Merge DataFrames ---
df_merged = pd.merge(
    df_dim_content,
    df_agg_performance,
    on=['client_hash_id', 'content_hash_id'],
    how='left'
)

# Handle cases where content might not have performance data
# Fill numerical columns with 0 and avg_position with the mean
df_merged['total_impressions'] = df_merged['total_impressions'].fillna(0)
df_merged['total_clicks'] = df_merged['total_clicks'].fillna(0)
df_merged['avg_position'] = df_merged['avg_position'].fillna(df_merged['avg_position'].mean())

# For latest_report_date, use content_updated_date if no performance data exists, or current timestamp if content_updated_date is also missing
df_merged['latest_report_date'] = df_merged['latest_report_date'].fillna(df_merged['content_updated_date'])
df_merged['latest_report_date'] = df_merged['latest_report_date'].fillna(pd.Timestamp.now())

print(f"Shape of df_merged after merge: {df_merged.shape}")
print("df_merged head:")
display(df_merged.head())

NameError: name 'pd' is not defined

In [2]:
# --- 4. Calculate Scores ---

# Staleness Score
# Days since content was last updated relative to the latest report date for that content
df_merged['days_since_update'] = (df_merged['latest_report_date'] - df_merged['content_updated_date']).dt.days

# Normalize staleness: higher days = higher score
max_days_since_update = df_merged['days_since_update'].max()
df_merged['staleness_score'] = df_merged['days_since_update'] / max_days_since_update

# CTR Opportunity Score
# Calculate CTR, handle division by zero for impressions
df_merged['CTR'] = np.where(
    df_merged['total_impressions'] > 0,
    df_merged['total_clicks'] / df_merged['total_impressions'],
    0
)

# Normalize components for CTR opportunity
# Impressions: higher is better
max_impressions = df_merged['total_impressions'].max()
df_merged['norm_impressions'] = df_merged['total_impressions'] / max_impressions if max_impressions > 0 else 0

# Low CTR: 1 - CTR (higher if CTR is low)
df_merged['low_ctr_factor'] = 1 - df_merged['CTR']

# Avg Position: lower is better (so invert or use 1 - (pos/max_pos))
# Capped at a reasonable max position to avoid skewing by extremely high positions
max_eff_position = df_merged['avg_position'].max()
df_merged['norm_avg_position_inv'] = 1 - (df_merged['avg_position'] / max_eff_position) if max_eff_position > 0 else 0

# Combine factors for CTR opportunity score
# This formula emphasizes pages with high impressions, good position (low value), and low CTR (high 1-CTR value)
# We'll multiply these factors. If impressions are 0, this should be 0.
df_merged['ctr_opportunity_score'] = (
    df_merged['norm_impressions'] *
    df_merged['low_ctr_factor'] *
    df_merged['norm_avg_position_inv']
)

# Handle potential NaN if any normalisation had max_val = 0 for all entries (e.g. all 0 impressions)
df_merged['ctr_opportunity_score'] = df_merged['ctr_opportunity_score'].fillna(0)

# Baseline Action Score
# Weighted sum of staleness and CTR opportunity scores
# Weights can be adjusted based on business priorities
weight_staleness = 0.5
weight_ctr_opportunity = 0.5
df_merged['baseline_action_score'] = (
    (df_merged['staleness_score'] * weight_staleness) +
    (df_merged['ctr_opportunity_score'] * weight_ctr_opportunity)
)

print("DataFrame with calculated scores:")
display(df_merged[['client_hash_id', 'content_hash_id', 'content_updated_date', 'latest_report_date', 'days_since_update', 'total_impressions', 'CTR', 'staleness_score', 'ctr_opportunity_score', 'baseline_action_score']].head(10))

NameError: name 'df_merged' is not defined

In [3]:
# --- 5. Rank Content Items ---
df_merged['rank'] = df_merged['baseline_action_score'].rank(ascending=False)

# --- 6. Assign Reason Codes ---
# Define thresholds for reason codes
STALE_THRESHOLD_DAYS = 200 # Content older than 200 days is considered stale
LOW_CTR_OPPORTUNITY_THRESHOLD = 0.01 # A low CTR indicates opportunity if other conditions met
AVG_POSITION_GOOD_THRESHOLD = 15 # Position better than 15 is considered 'reasonable'
IMPRESSIONS_THRESHOLD = 1000 # Minimum impressions for CTR opportunity to be relevant

def assign_reason_code(row):
    is_stale = row['days_since_update'] > STALE_THRESHOLD_DAYS
    is_low_ctr = row['CTR'] < LOW_CTR_OPPORTUNITY_THRESHOLD
    has_impressions = row['total_impressions'] >= IMPRESSIONS_THRESHOLD
    has_reasonable_pos = row['avg_position'] <= AVG_POSITION_GOOD_THRESHOLD

    if is_stale and is_low_ctr and has_impressions and has_reasonable_pos:
        return 'STALE_AND_LOW_CTR'
    elif is_stale:
        return 'STALE_CONTENT'
    elif is_low_ctr and has_impressions and has_reasonable_pos:
        return 'LOW_CTR_OPPORTUNITY'
    else:
        return 'NONE_APPLICABLE'

df_merged['reason_code'] = df_merged.apply(assign_reason_code, axis=1)

# Sort by rank for the final queue
df_ranked_queue = df_merged.sort_values(by='rank', ascending=True).reset_index(drop=True)

print("Ranked Queue with Reason Codes (Top 10):")
display(df_ranked_queue[['client_hash_id', 'content_hash_id', 'baseline_action_score', 'rank', 'reason_code', 'days_since_update', 'CTR', 'total_impressions', 'avg_position']].head(10))

NameError: name 'df_merged' is not defined

In [4]:
# --- 7. Write to CSV ---
import os

output_dir = 'work/outputs'
os.makedirs(output_dir, exist_ok=True)

output_filepath = os.path.join(output_dir, 'baseline_action_score.csv')
df_ranked_queue.to_csv(output_filepath, index=False)

print(f"Ranked queue saved to: {output_filepath}")

NameError: name 'df_ranked_queue' is not defined

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.